# 2b - The client portfolio, built by hand

**Time:** 35 minutes  |  **AI: OFF** (no Gemini, no ChatGPT in this notebook)

**What you will do**
- Build two functions: `portfolio_return` and `portfolio_vol`.
- Add a guard that stops the code when the weights do not sum to 1.
- Check your code with a known case (all correlations 1), then run a crisis (all correlations 0.9).

Each blank is marked `# YOUR CODE`. Replace the `...`. After each function, run the test cell and compare with the number written in the text.

## The problem

Funds A, B and C. The client holds **20% in A, 30% in B and 50% in C**. What are the return and the risk of the portfolio? How much does diversification save? What happens in a crisis, when all funds move together?

| Step | What | You should get |
|---|---|---|
| 1 | `portfolio_return(weights, returns)` | 7.6% |
| 2 | `portfolio_vol(weights, vol, corr)` | 18.44%, against 21.5% for the weighted average of the vols |
| 3 | the guard: weights must sum to 1 | a clear error message |
| 4 | known case: all correlations 1 | exactly 21.5% |
| 5 | crisis: all correlations 0.9 | 20.88% |

In [ ]:
import numpy as np

# The assumptions. Rates and weights are decimals.
w = np.array([0.20, 0.30, 0.50])      # client weights in A, B, C
ret = np.array([0.05, 0.07, 0.09])    # expected returns
vol = np.array([0.15, 0.20, 0.25])    # volatilities
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])

## Step 1 - Portfolio return

The return of the portfolio is each weight times its return, added up. Use `np.sum` and a vectorized operation (or `@`).

You should get **7.6%**.

In [ ]:
def portfolio_return(weights, returns):
    return ...   # YOUR CODE: weights times returns, added up

In [ ]:
print(f"Portfolio return: {portfolio_return(w, ret):.1%}")

## Step 2 - Portfolio volatility

Three lines, the same as in 2a:
1. the covariance matrix: `np.diag(vol) @ corr @ np.diag(vol)`
2. the variance: `weights @ cov @ weights`
3. the volatility: the square root of the variance, `np.sqrt`

You should get **18.44%**. The weighted average of the vols is 21.5%: diversification saves about 3.1 points.

(`:.2%` in an f-string shows a decimal as a percent: 0.1844 becomes 18.44%.)

In [ ]:
def portfolio_vol(weights, vol, corr):
    cov = ...   # YOUR CODE: the covariance matrix
    var = ...   # YOUR CODE: the variance
    return ...  # YOUR CODE: the volatility

In [ ]:
avg_vol = np.sum(w * vol)   # weighted average of the vols
print(f"Portfolio vol:    {portfolio_vol(w, vol, corr):.2%}")
print(f"Weighted average: {avg_vol:.2%}")

## Step 3 - The guard: weights must sum to 1

If the weights do not add up to 1, the result means nothing. A guard stops the code with a message in finance words.

One trap: computers store most decimals with tiny errors. Run the cell below.

**Your prediction:** does `0.7 + 0.2 + 0.1` give exactly 1?

In [ ]:
print(0.7 + 0.2 + 0.1)
print(0.7 + 0.2 + 0.1 == 1)
print(round(0.7 + 0.2 + 0.1, 6) == 1)

It gives `0.9999999999999999`, so `== 1` is False. **Never test decimals with `==`**. Round first: `round(np.sum(weights), 6) != 1` is True only when the weights really do not sum to 1.

Add the guard at the top of `portfolio_vol`, then keep the three lines of step 2.

In [ ]:
def portfolio_vol(weights, vol, corr):
    if ...:   # YOUR CODE: the condition round(np.sum(weights), 6) != 1
        raise ValueError("Weights must sum to 1.")
    cov = ...   # YOUR CODE: the same three lines as in step 2
    var = ...
    return ...

In [ ]:
# Test 1: the client portfolio still works. You should get 18.44%
print(f"{portfolio_vol(w, vol, corr):.2%}")

**Test 2 fails on purpose.** The weights are 50%, 30% and 30%: they sum to 110%. Read the message: last line first.

**Your prediction:** which error type will Python show, and which message?

In [ ]:
portfolio_vol(np.array([0.5, 0.3, 0.3]), vol, corr)

`ValueError: Weights must sum to 1.` This is **your** message, raised by your guard. This error is intended.

## Step 4 - Known case: all correlations 1

A **known case** is a situation where you know the answer without the code. If all correlations are 1, the funds move in exact lockstep and nothing cancels: the portfolio vol must be the weighted average of the vols, **21.5%**, exactly.

Make a 3x3 array of ones with `np.ones((3, 3))` and run your function on it. If you do not get 21.5%, the code is wrong. (Checks card, entry 5.)

In [ ]:
corr_one = ...   # YOUR CODE: a 3x3 array full of ones
print(f"All correlations 1: {portfolio_vol(w, vol, corr_one):.2%}")

## Step 5 - Crisis: all correlations 0.9

In a crisis, funds fall together. Build a correlation matrix with 0.9 everywhere, then put 1 on the diagonal (a fund is always perfectly correlated with itself).

- `np.full((3, 3), 0.9)` makes a 3x3 array full of 0.9.
- `np.fill_diagonal(crisis, 1.0)` sets the diagonal to 1.

You should get **20.88%**, almost back to the weighted average of 21.5%. Diversification disappears when you need it most.

In [ ]:
crisis = ...   # YOUR CODE: a 3x3 array full of 0.9
...            # YOUR CODE: put 1.0 on the diagonal (np.fill_diagonal)

print(crisis)
print(f"Crisis vol: {portfolio_vol(w, vol, crisis):.2%}")

## What you learned

**Python**
- **Function** on arrays: `portfolio_return` and `portfolio_vol` take **arrays** and give back a number. One formula, any portfolio.
- **Matrix product** (`weights @ cov @ weights`), **Diagonal matrix** (`np.diag(vol)`) and **Square root** (`np.sqrt`) inside the function.
- **Guard**: `if round(np.sum(weights), 6) != 1: raise ValueError(...)` stops the code with your message. **round** first: never test decimals with `==`.
- **Known case**: `np.ones((3, 3))` for correlations. **2D array** built with `np.full((3, 3), 0.9)` and `np.fill_diagonal`.

**Finance**
- The client portfolio: return 7.6%, vol 18.44% against 21.5% for the weighted average of the vols.
- Known case: all correlations 1 gives exactly the weighted average, 21.5%. If it does not, the code is wrong.
- Crisis: all correlations 0.9 gives 20.88%. Diversification disappears when you need it most.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right